In [ ]:
FW_DIR = "../fw/examples/cw305_driver_example"
BIN_FILE = f"{FW_DIR}/cw305_driver_example.bin"
BOOTLOADER = "python3 ../../sdk/toolchain/bootloader.py"
BUFF_LEN = 22
BS_FILE = "../vivado/risqrypt_cw305.runs/impl_1/fpga_top.bit"
VERBOSE = True

In [2]:
import chipwhisperer as cw
scope = cw.scope()
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"

In [3]:
TARGET_PLATFORM = 'CW305_100t'

In [4]:
scope.gain.db = 25
platform = 'cw305'
fpga_id = '100t'

target = cw.target(scope, cw.targets.CW305, force=False, fpga_id=fpga_id, platform=platform, bsfile=BS_FILE)

(ChipWhisperer NAEUSB WARNING|File naeusb.py:800) Your firmware (0.51) is outdated - latest is 0.53 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information


In [5]:
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
target.pll.pll_outfreq_set(20E6, 1)

# 1ms is plenty of idling time
target.clkusbautooff = True
target.clksleeptime = 1

In [6]:
%%bash -s "$FW_DIR" "$BUFF_LEN"
cd $1
make USE_DONE=1 BUFF_LEN=$2

-DUSE_DONE -DBUFF_LEN=22 -O0 -I../../driver -I../../../../sdk/lib -march=rv32im -mabi=ilp32 -mstrict-align -T ../../../../sdk/toolchain/linksc.ld -flto -ffunction-sections -fdata-sections -nostartfiles -o cw305_driver_example.elf
/opt/riscv/bin/riscv64-unknown-elf-gcc cw305_driver_example.c ../../driver/cw305.c ../../../../sdk/toolchain/start.s ../../../../sdk/lib/util.c ../../../../sdk/lib/uart.c ../../../../sdk/lib/timer.c ../../../../sdk/lib/gpio.c ../../../../sdk/lib/keccak.c ../../../../sdk/lib/ntt_lite.c ../../../../sdk/lib/x2x.c -DUSE_DONE -DBUFF_LEN=22 -O0 -I../../driver -I../../../../sdk/lib -march=rv32im -mabi=ilp32 -mstrict-align -T ../../../../sdk/toolchain/linksc.ld -flto -ffunction-sections -fdata-sections -nostartfiles -o cw305_driver_example.elf
/opt/riscv/bin/riscv64-unknown-elf-objcopy -O binary -j .init -j .text -j .rodata -j .data cw305_driver_example.elf cw305_driver_example.bin
python3 ../../../../sdk/toolchain/rom_generator.py cw305_driver_example.bin cw305_drive

In [ ]:
%%bash -s "$BIN_FILE" "$BOOTLOADER"
$2 -f $1 -q

Successfully connected to /dev/ttyUSB0
Sent: -p
Waiting for opcodes...
Sending file: ../fw/examples/cw305_driver_example/cw305_driver_example.bin


../fw/examples/cw305_driver_example/cw305_driver_example.bin: 100%|██████████| 61.6k/61.6k [01:03<00:00, 972B/s]  


File transmission complete.
Serial port closed.


In [ ]:
import time
time.sleep(3)

In [ ]:
import tqdm
import random

hex_len = BUFF_LEN

for i in tqdm.tqdm(range(128)):
    message = bytes([random.randint(0, 255) for _ in range(hex_len//2 + 1)]).hex()[:hex_len]
    if VERBOSE:
        print("Sending message {}".format(message))
    target.fpga_write(0, message)
    while(True):
        status = target.fpga_read(1, 1)
        if status[0] == 0xff:
            break
    s = target.fpga_read(0, len(message))
    if VERBOSE:
        print(s)
    s_ = s.decode('utf-8')
    assert s_ == message, "Mismatch: sent {}, received {}".format(message, s_)
    if VERBOSE:
        print("Read back: {}".format(s.decode('utf-8')))
        print()
        print()

100%|██████████| 128/128 [00:09<00:00, 13.18it/s]


In [13]:
if scope._is_husky:
    scope.clock.clkgen_freq = 40e6
    scope.clock.clkgen_src = 'extclk'
    scope.clock.adc_mul = 2
    # if the target PLL frequency is changed, the above must also be changed accordingly
else:
    scope.clock.adc_src = "extclk_x4"

In [14]:
import time
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break 
assert (scope.clock.adc_locked), "ADC failed to lock"

In [15]:
%%bash -s "$FW_DIR" "$BUFF_LEN"
cd $1
make BUFF_LEN=$2

-DBUFF_LEN=22 -O0 -I../../driver -I../../../../sdk/lib -march=rv32im -mabi=ilp32 -mstrict-align -T ../../../../sdk/toolchain/linksc.ld -flto -ffunction-sections -fdata-sections -nostartfiles -o cw305_driver_example.elf
/opt/riscv/bin/riscv64-unknown-elf-gcc cw305_driver_example.c ../../driver/cw305.c ../../../../sdk/toolchain/start.s ../../../../sdk/lib/util.c ../../../../sdk/lib/uart.c ../../../../sdk/lib/timer.c ../../../../sdk/lib/gpio.c ../../../../sdk/lib/keccak.c ../../../../sdk/lib/ntt_lite.c ../../../../sdk/lib/x2x.c -DBUFF_LEN=22 -O0 -I../../driver -I../../../../sdk/lib -march=rv32im -mabi=ilp32 -mstrict-align -T ../../../../sdk/toolchain/linksc.ld -flto -ffunction-sections -fdata-sections -nostartfiles -o cw305_driver_example.elf
/opt/riscv/bin/riscv64-unknown-elf-objcopy -O binary -j .init -j .text -j .rodata -j .data cw305_driver_example.elf cw305_driver_example.bin
python3 ../../../../sdk/toolchain/rom_generator.py cw305_driver_example.bin cw305_driver_example.mem 
/opt/ri

In [ ]:
%%bash -s "$BIN_FILE" "$BOOTLOADER"
$2 -f $1 -q

Successfully connected to /dev/ttyUSB0
Sent: -p
Waiting for opcodes...
Sending file: ../fw/examples/cw305_driver_example/cw305_driver_example.bin


../fw/examples/cw305_driver_example/cw305_driver_example.bin: 100%|██████████| 61.6k/61.6k [01:03<00:00, 972B/s]  


File transmission complete.
Serial port closed.


In [17]:
import time
time.sleep(3)

In [18]:
import tqdm
import random

hex_len = BUFF_LEN

for i in tqdm.tqdm(range(128)):
    message = bytes([random.randint(0, 255) for _ in range(hex_len//2 + 1)]).hex()[:hex_len]
    if VERBOSE:
        print("Sending message {}".format(message))
    scope.arm()
    if VERBOSE:
        print("Sending message {}".format(message))
    target.fpga_write(0, message)
    ret = scope.capture(poll_done=True)
    s = target.fpga_read(0, len(message))
    if VERBOSE:
        print(s)
    s_ = s.decode('utf-8')
    assert s_ == message, "Mismatch: sent {}, received {}".format(message, s_)
    if VERBOSE:
        print("Read back: {}".format(s.decode('utf-8')))
        print()
        print()

100%|██████████| 128/128 [00:09<00:00, 13.15it/s]
